In [9]:
##### Final cleaning of RF training data before runs
# convert to log, remove outliers, add country codes 

import os
import pandas as pd
from pathlib import Path
import numpy as np
import geopandas as gpd
from functools import reduce

In [10]:
### Set-up

# Get the current working directory
cd = Path.cwd().parent.parent 

# data 
capital_abs = pd.read_csv(f'{cd}/Data/Clean/Training_data/capital_absolute.csv')
labor_abs = pd.read_csv(f'{cd}/Data/Clean/Training_data/labor_absolute.csv')

capital_rtv = pd.read_csv(f'{cd}/Data/Clean/Training_data/capital_relative.csv')
labor_rtv = pd.read_csv(f'{cd}/Data/Clean/Training_data/labor_relative.csv')

# save paths
save_path_capital_abs = f'{cd}/Data/Clean/Training_data/capital_absolute_final.csv'
save_path_labor_abs = f'{cd}/Data/Clean/Training_data/labor_absolute_final.csv'

save_path_capital_rtv = f'{cd}/Data/Clean/Training_data/capital_relative_final.csv'
save_path_labor_rtv = f'{cd}/Data/Clean/Training_data/labor_relative_final.csv'

In [11]:
##### Variables 

### Drop 1 from % variables (so don't sum to 100)
# one of each of field size %'s and production mix %'s 
# share_vlarge_field, other_share_production_USD  

col_to_drop = ['share_vlarge_field', 'other_share_production_USD']

capital_abs = capital_abs.drop(columns = col_to_drop)
labor_abs = labor_abs.drop(columns = col_to_drop)

col_to_drop = ['rtv_share_vlarge_field', 'rtv_other_share_production_USD']

capital_rtv = capital_rtv.drop(columns = col_to_drop)
labor_rtv = labor_rtv.drop(columns = col_to_drop)

### Drop aggregate variables

capital_abs = capital_abs.drop(columns = ['GEO_ID_NAME', 'ag_capital_stock_USD_nominal', 'total_production_USD', 'total_production_t'])
labor_abs = labor_abs.drop(columns = ['GEO_ID_NAME', 'ag_jobs', 'total_production_USD', 'total_production_t'])

In [12]:
##### Drop regions with missing data 

c_len_pre = len(capital_abs)
capital_abs = capital_abs.dropna()
c_len_pos = len(capital_abs)
c_dropped = c_len_pre - c_len_pos

print(f'Capital absolute: {c_dropped} out of {c_len_pre} dropped')

c_len_pre = len(capital_rtv)
capital_rtv = capital_rtv.dropna()
c_len_pos = len(capital_rtv)
c_dropped = c_len_pre - c_len_pos

print(f'Capital relative: {c_dropped} out of {c_len_pre} dropped')

l_len_pre = len(labor_abs)
labor_abs = labor_abs.dropna()
l_len_pos = len(labor_abs)
l_dropped = l_len_pre - l_len_pos

print(f'Labor absolute: {l_dropped} out of {l_len_pre} dropped')

l_len_pre = len(labor_rtv)
labor_rtv = labor_rtv.dropna()
l_len_pos = len(labor_rtv)
l_dropped = l_len_pre - l_len_pos

print(f'Labor relative: {l_dropped} out of {l_len_pre} dropped')

Capital absolute: 371 out of 10482 dropped
Capital relative: 2 out of 10482 dropped
Labor absolute: 469 out of 11500 dropped
Labor relative: 2 out of 11500 dropped


In [152]:
# #####  Drop extreme values (top/bottom 1%)

# lower_cap = capital['capital_intensity_USD_per_USD'].quantile(0.01)
# upper_cap = capital['capital_intensity_USD_per_USD'].quantile(0.99)

# capital = capital[
#     (capital['capital_intensity_USD_per_USD'] >= lower_cap) &
#     (capital['capital_intensity_USD_per_USD'] <= upper_cap)
# ]

# lower_lab = labor['labor_intensity_jobs_per_million_USD'].quantile(0.01)
# upper_lab = labor['labor_intensity_jobs_per_million_USD'].quantile(0.99)

# labor = labor[
#     (labor['labor_intensity_jobs_per_million_USD'] >= lower_lab) &
#     (labor['labor_intensity_jobs_per_million_USD'] <= upper_lab)
# ]

In [13]:
##### Add country ID column

capital_abs['country_ID'] = capital_abs['PROJ_ID'].str[:3]
labor_abs['country_ID'] = labor_abs['PROJ_ID'].str[:3]

capital_rtv['country_ID'] = capital_rtv['PROJ_ID'].str[:3]
labor_rtv['country_ID'] = labor_rtv['PROJ_ID'].str[:3]

In [14]:
##### Convert intensities to log for absolute model 

# define function fof log transform
def log_transform(df, cols, eps_cols=None):
    eps_cols = eps_cols or []

    for col in cols:
        if col in eps_cols:
            df[f"log_{col}"] = np.log(df[col] + 1e-6)
        else:
            df[f"log_{col}"] = np.log(df[col])

    df.drop(columns=cols, inplace=True)
    return df

capital_cols = [
    "capital_intensity_USD_per_USD",
    "capital_intensity_USD_per_tonne",
    "tonnes_production_per_HA",
    "USD_production_per_HA",
    "country_capital_intensity_USD_per_USD",
    "country_labor_intensity_jobs_per_million_USD",
    "country_capital_intensity_USD_per_tonne",
    "country_labor_intensity_jobs_per_tonne"
]

labor_cols = [
    "labor_intensity_jobs_per_million_USD",
    "labor_intensity_jobs_per_tonne",
    "tonnes_production_per_HA",
    "USD_production_per_HA",
    "country_labor_intensity_jobs_per_million_USD",
    "country_capital_intensity_USD_per_USD",
    "country_capital_intensity_USD_per_tonne",
    "country_labor_intensity_jobs_per_tonne"
]

eps_cols = [
    "tonnes_production_per_HA",
    "USD_production_per_HA"
]

capital_abs = log_transform(capital_abs, capital_cols, eps_cols)
labor_abs = log_transform(labor_abs, labor_cols, eps_cols)

In [15]:
# ### OPTIONAL: drop countries from capital where proxy was used 

# count_countries = ['BRA', 'EGY', 'GHA', 'IND', 'TUR', 'ZAF', 'TZA', 'ARG']

# capital = capital[~capital['country_ID'].isin(count_countries)]

count_countries = ['ZAF']
capital_abs = capital_abs[~capital_abs['country_ID'].isin(count_countries)]
capital_rtv = capital_rtv[~capital_rtv['country_ID'].isin(count_countries)]

In [16]:
##### Save 
capital_abs.to_csv(save_path_capital_abs, index=False)
labor_abs.to_csv(save_path_labor_abs, index=False)

capital_rtv.to_csv(save_path_capital_rtv, index=False)
labor_rtv.to_csv(save_path_labor_rtv, index=False)